# 📄 Sentence Tokenization - Essential for RAG Document Chunking

## What is Sentence Tokenization?

**Sentence tokenization (sentence segmentation) splits text into individual sentences.**

### Example:
```python
text = "Hello! How are you? I'm doing great."
sentences = ["Hello!", "How are you?", "I'm doing great."]
```

## Why is This CRITICAL for RAG? 🎯

### Document Chunking for Retrieval:

1. **Context Windows**: Models have token limits (512-4096)
2. **Semantic Units**: Sentences preserve meaning
3. **Retrieval Granularity**: Better than random chunks
4. **Answer Precision**: Find exact relevant context

### The RAG Pipeline:
```
Long Document
    ↓ Sentence Tokenization
Individual Sentences
    ↓ Embedding
Vector Database
    ↓ Retrieval
Relevant Context → LLM → Answer
```

## Challenges in Sentence Tokenization:

❌ **Not all periods end sentences:**
- "Dr. Smith works at A.I. Corp." (abbreviations)
- "It costs $99.99 in the U.S." (numbers, acronyms)
- "He said \"Hello.\" Then he left." (quotes)

✅ **Good sentence tokenizers handle:**
- Abbreviations (Dr., Mr., Ph.D.)
- Numbers and decimals (3.14, $19.99)
- Quotations and dialogue
- Multiple languages
- Edge cases

## We'll Cover:

1. Simple regex-based splitting
2. NLTK sentence tokenizer
3. spaCy sentence segmentation
4. Language-specific tokenizers
5. RAG-optimized chunking strategies
6. Handling edge cases

## 1. Simple Sentence Splitting (Naive Approach)

In [1]:
import re

def simple_sentence_split(text):
    """Simple split on periods, question marks, exclamation marks"""
    # WARNING: This is too naive for production!
    sentences = re.split(r'[.!?]+', text)
    return [s.strip() for s in sentences if s.strip()]

# Test
text = "Hello! How are you? I'm doing great. What about you?"
sentences = simple_sentence_split(text)

print(f"Text: {text}")
print(f"\nSentences ({len(sentences)}):")
for i, sent in enumerate(sentences, 1):
    print(f"{i}. {sent}")

print("\n✅ Works for simple cases!")

Text: Hello! How are you? I'm doing great. What about you?

Sentences (4):
1. Hello
2. How are you
3. I'm doing great
4. What about you

✅ Works for simple cases!


In [2]:
# But fails on edge cases
tricky_text = "Dr. Smith works at A.I. Corp. It's located at 123 Main St. in the U.S."

naive_split = simple_sentence_split(tricky_text)

print(f"Tricky text: {tricky_text}")
print(f"\nNaive split ({len(naive_split)} sentences):")
for i, sent in enumerate(naive_split, 1):
    print(f"{i}. {sent}")

print("\n❌ Splits on abbreviations! We need better tools.")

Tricky text: Dr. Smith works at A.I. Corp. It's located at 123 Main St. in the U.S.

Naive split (7 sentences):
1. Dr
2. Smith works at A
3. I
4. Corp
5. It's located at 123 Main St
6. in the U
7. S

❌ Splits on abbreviations! We need better tools.


## 2. NLTK Sentence Tokenizer (Recommended)

In [3]:
# Install NLTK
# !pip install nltk

import nltk
from nltk.tokenize import sent_tokenize

# Download the sentence tokenizer model
nltk.download('punkt', quiet=True)

print("✅ NLTK Punkt tokenizer loaded")

✅ NLTK Punkt tokenizer loaded


In [4]:
# Test on tricky text
tricky_text = "Dr. Smith works at A.I. Corp. It's located at 123 Main St. in the U.S."

nltk_sentences = sent_tokenize(tricky_text)

print(f"Text: {tricky_text}")
print(f"\nNLTK split ({len(nltk_sentences)} sentences):")
for i, sent in enumerate(nltk_sentences, 1):
    print(f"{i}. {sent}")

print("\n✅ Correctly handles abbreviations!")

Text: Dr. Smith works at A.I. Corp. It's located at 123 Main St. in the U.S.

NLTK split (3 sentences):
1. Dr. Smith works at A.I.
2. Corp.
3. It's located at 123 Main St. in the U.S.

✅ Correctly handles abbreviations!


## 3. spaCy Sentence Segmentation

In [5]:
# Install spaCy
# !pip install spacy
# !python -m spacy download en_core_web_sm

import spacy

# Load English model
nlp = spacy.load("en_core_web_sm")

# Process text
doc = nlp(tricky_text)

# Extract sentences
spacy_sentences = [sent.text for sent in doc.sents]

print(f"Text: {tricky_text}")
print(f"\nspaCy split ({len(spacy_sentences)} sentences):")
for i, sent in enumerate(spacy_sentences, 1):
    print(f"{i}. {sent}")

print("\n✅ spaCy also handles it well!")

Text: Dr. Smith works at A.I. Corp. It's located at 123 Main St. in the U.S.

spaCy split (2 sentences):
1. Dr. Smith works at A.I. Corp.
2. It's located at 123 Main St. in the U.S.

✅ spaCy also handles it well!


## 4. Comparing Methods on Complex Text

In [6]:
# Complex RAG document
complex_text = """Retrieval-Augmented Generation (RAG) improves AI systems. Dr. Johnson's 
research at M.I.T. shows 40% accuracy gains. The system costs $50K-$100K to deploy. 
"This is revolutionary!" said Prof. Smith, Ph.D. Can it handle edge cases? Yes! It works 
with URLs like https://example.com and emails like test@example.com."""

# Method 1: Simple split
simple_sents = simple_sentence_split(complex_text)

# Method 2: NLTK
nltk_sents = sent_tokenize(complex_text)

# Method 3: spaCy
doc = nlp(complex_text)
spacy_sents = [sent.text.strip() for sent in doc.sents]

print("Complex Text Tokenization Comparison:\n")
print(f"Text: {complex_text[:100]}...\n")

print(f"Simple split: {len(simple_sents)} sentences")
print(f"NLTK: {len(nltk_sents)} sentences")
print(f"spaCy: {len(spacy_sents)} sentences")

print("\nNLTK results:")
for i, sent in enumerate(nltk_sents, 1):
    print(f"{i}. {sent[:80]}..." if len(sent) > 80 else f"{i}. {sent}")

Complex Text Tokenization Comparison:

Text: Retrieval-Augmented Generation (RAG) improves AI systems. Dr. Johnson's 
research at M.I.T. shows 40...

Simple split: 16 sentences
NLTK: 8 sentences
spaCy: 6 sentences

NLTK results:
1. Retrieval-Augmented Generation (RAG) improves AI systems.
2. Dr. Johnson's 
research at M.I.T.
3. shows 40% accuracy gains.
4. The system costs $50K-$100K to deploy.
5. "This is revolutionary!"
6. said Prof. Smith, Ph.D. Can it handle edge cases?
7. Yes!
8. It works 
with URLs like https://example.com and emails like test@example.com.


## 5. RAG-Optimized Chunking Strategy

In [7]:
from transformers import AutoTokenizer

def chunk_by_sentences(text, max_tokens=512, tokenizer_name='bert-base-uncased'):
    """
    Chunk text into sentence-based chunks within token limit.
    
    This is the RECOMMENDED approach for RAG!
    """
    # Initialize tokenizer
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    
    # Split into sentences
    sentences = sent_tokenize(text)
    
    chunks = []
    current_chunk = []
    current_tokens = 0
    
    for sentence in sentences:
        # Count tokens in sentence
        sentence_tokens = len(tokenizer.encode(sentence, add_special_tokens=False))
        
        # If adding this sentence exceeds limit, start new chunk
        if current_tokens + sentence_tokens > max_tokens and current_chunk:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentence]
            current_tokens = sentence_tokens
        else:
            current_chunk.append(sentence)
            current_tokens += sentence_tokens
    
    # Add remaining chunk
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    
    return chunks

# Test on long document
long_doc = """Retrieval-Augmented Generation is transforming AI. It combines retrieval 
with generation. This approach improves accuracy. It reduces hallucinations. The system 
retrieves relevant documents first. Then it uses them as context. The language model 
generates better responses. This is widely used in production. Companies love the results. 
RAG systems are here to stay."""

chunks = chunk_by_sentences(long_doc, max_tokens=30)

print(f"Original document: {len(long_doc)} chars\n")
print(f"Created {len(chunks)} chunks:\n")

for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i} ({len(chunk)} chars):")
    print(f"  {chunk}\n")

Original document: 370 chars

Created 3 chunks:

Chunk 1 (150 chars):
  Retrieval-Augmented Generation is transforming AI. It combines retrieval 
with generation. This approach improves accuracy. It reduces hallucinations.

Chunk 2 (160 chars):
  The system 
retrieves relevant documents first. Then it uses them as context. The language model 
generates better responses. This is widely used in production.

Chunk 3 (57 chars):
  Companies love the results. RAG systems are here to stay.



## 6. Chunking with Overlap (Best Practice)

In [8]:
def chunk_with_overlap(text, max_tokens=512, overlap_sentences=1, 
                       tokenizer_name='bert-base-uncased'):
    """
    Chunk with sentence overlap for context preservation.
    
    Overlap helps maintain context across chunks!
    """
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    sentences = sent_tokenize(text)
    
    chunks = []
    i = 0
    
    while i < len(sentences):
        current_chunk = []
        current_tokens = 0
        
        # Build chunk
        j = i
        while j < len(sentences):
            sent_tokens = len(tokenizer.encode(sentences[j], add_special_tokens=False))
            
            if current_tokens + sent_tokens > max_tokens and current_chunk:
                break
            
            current_chunk.append(sentences[j])
            current_tokens += sent_tokens
            j += 1
        
        if current_chunk:
            chunks.append(' '.join(current_chunk))
        
        # Move forward with overlap
        i = j - overlap_sentences if j > i + overlap_sentences else j
        if i >= len(sentences):
            break
    
    return chunks

# Test overlap chunking
overlap_chunks = chunk_with_overlap(long_doc, max_tokens=30, overlap_sentences=1)

print(f"Chunking with 1-sentence overlap:\n")
print(f"Created {len(overlap_chunks)} chunks:\n")

for i, chunk in enumerate(overlap_chunks, 1):
    print(f"Chunk {i}:")
    print(f"  {chunk}\n")

print("💡 Notice: Chunks share sentences for better context!")

Chunking with 1-sentence overlap:

Created 4 chunks:

Chunk 1:
  Retrieval-Augmented Generation is transforming AI. It combines retrieval 
with generation. This approach improves accuracy. It reduces hallucinations.

Chunk 2:
  It reduces hallucinations. The system 
retrieves relevant documents first. Then it uses them as context. The language model 
generates better responses.

Chunk 3:
  The language model 
generates better responses. This is widely used in production. Companies love the results. RAG systems are here to stay.

Chunk 4:
  RAG systems are here to stay.

💡 Notice: Chunks share sentences for better context!


## 7. Multilingual Sentence Tokenization

In [9]:
# NLTK supports multiple languages
multilingual_texts = [
    ("English", "This is English. It has two sentences."),
    ("Spanish", "Este es español. Tiene dos oraciones."),
    ("French", "C'est français. Il a deux phrases."),
    ("German", "Das ist Deutsch. Es hat zwei Sätze."),
]

print("Multilingual Sentence Tokenization:\n")

for lang, text in multilingual_texts:
    # NLTK auto-detects language in many cases
    sentences = sent_tokenize(text)
    print(f"{lang}:")
    print(f"  Text: {text}")
    print(f"  Sentences: {sentences}\n")

print("✅ NLTK handles multiple languages!")

Multilingual Sentence Tokenization:

English:
  Text: This is English. It has two sentences.
  Sentences: ['This is English.', 'It has two sentences.']

Spanish:
  Text: Este es español. Tiene dos oraciones.
  Sentences: ['Este es español.', 'Tiene dos oraciones.']

French:
  Text: C'est français. Il a deux phrases.
  Sentences: ["C'est français.", 'Il a deux phrases.']

German:
  Text: Das ist Deutsch. Es hat zwei Sätze.
  Sentences: ['Das ist Deutsch.', 'Es hat zwei Sätze.']

✅ NLTK handles multiple languages!


## 8. Handling Special Cases

In [10]:
# Edge cases
edge_cases = [
    "Emails: Contact me at user@example.com. I'll respond quickly.",
    "URLs: Visit https://example.com. It has great content.",
    "Numbers: The value is 3.14159. That's pi!",
    "Quotes: He said \"Hello.\" She replied \"Hi.\"",
    "Lists: Items: 1. First 2. Second 3. Third. That's all.",
]

print("Handling Edge Cases:\n")

for text in edge_cases:
    sentences = sent_tokenize(text)
    print(f"Text: {text}")
    print(f"Sentences ({len(sentences)}): {sentences}\n")

Handling Edge Cases:

Text: Emails: Contact me at user@example.com. I'll respond quickly.
Sentences (2): ['Emails: Contact me at user@example.com.', "I'll respond quickly."]

Text: URLs: Visit https://example.com. It has great content.
Sentences (2): ['URLs: Visit https://example.com.', 'It has great content.']

Text: Numbers: The value is 3.14159. That's pi!
Sentences (2): ['Numbers: The value is 3.14159.', "That's pi!"]

Text: Quotes: He said "Hello." She replied "Hi."
Sentences (2): ['Quotes: He said "Hello."', 'She replied "Hi."']

Text: Lists: Items: 1. First 2. Second 3. Third. That's all.
Sentences (5): ['Lists: Items: 1.', 'First 2.', 'Second 3.', 'Third.', "That's all."]



## 9. Real-World RAG Document Processing

In [11]:
# Realistic RAG document
rag_document = """Title: Understanding Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) is a powerful technique in modern NLP. It was 
introduced by Facebook AI Research in 2020. The core idea is simple yet effective.

First, the system retrieves relevant documents from a knowledge base. This is typically 
done using dense embeddings and vector similarity search. Popular embedding models include 
BERT, Sentence-BERT, and E5.

Second, the retrieved documents serve as context for a language model. Models like GPT-3.5, 
GPT-4, or T5 then generate responses based on this context. This significantly improves 
factual accuracy.

The benefits are clear: reduced hallucination, better factual accuracy, and the ability to 
update knowledge without retraining. RAG systems are now widely deployed in production.

For implementation, start with document chunking. Use sentence-based chunking with overlap. 
Embed each chunk using a suitable model. Store embeddings in a vector database like Pinecone, 
Weaviate, or Chroma.

At query time, embed the user's question. Retrieve top-k similar chunks. Pass them to the 
LLM as context. The LLM generates a response grounded in retrieved facts.

Common challenges include: chunk size optimization, retrieval precision, and handling 
multi-hop reasoning. Research continues to address these issues."""

# Process for RAG
print("RAG Document Processing Pipeline:\n")
print(f"Original document: {len(rag_document)} characters\n")

# Step 1: Split into sentences
sentences = sent_tokenize(rag_document)
print(f"Step 1: Split into {len(sentences)} sentences\n")

# Step 2: Create chunks
chunks = chunk_with_overlap(rag_document, max_tokens=100, overlap_sentences=2)
print(f"Step 2: Created {len(chunks)} chunks with overlap\n")

# Step 3: Display chunks
print("Chunks for embedding:")
print("="*80)
for i, chunk in enumerate(chunks, 1):
    print(f"\nChunk {i}:")
    print(chunk[:150] + "..." if len(chunk) > 150 else chunk)

print("\n💡 These chunks are ready for embedding and vector storage!")

RAG Document Processing Pipeline:

Original document: 1341 characters

Step 1: Split into 21 sentences

Step 2: Created 5 chunks with overlap

Chunks for embedding:

Chunk 1:
Title: Understanding Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) is a powerful technique in modern NLP. It was 
introduced by...

Chunk 2:
Popular embedding models include 
BERT, Sentence-BERT, and E5. Second, the retrieved documents serve as context for a language model. Models like GPT-...

Chunk 3:
The benefits are clear: reduced hallucination, better factual accuracy, and the ability to 
update knowledge without retraining. RAG systems are now w...

Chunk 4:
Store embeddings in a vector database like Pinecone, 
Weaviate, or Chroma. At query time, embed the user's question. Retrieve top-k similar chunks. Pa...

Chunk 5:
Common challenges include: chunk size optimization, retrieval precision, and handling 
multi-hop reasoning. Research continues to address these issues...

💡 These chunks

## 10. Advanced: Semantic Sentence Splitting

In [12]:
# Sometimes we want to split by semantic boundaries, not just sentences
def semantic_chunking(text, max_tokens=100, tokenizer_name='bert-base-uncased'):
    """
    Chunk by semantic units (paragraphs or sentence groups)
    """
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    
    # Split by paragraphs first
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    
    chunks = []
    
    for para in paragraphs:
        para_tokens = len(tokenizer.encode(para, add_special_tokens=False))
        
        if para_tokens <= max_tokens:
            # Paragraph fits, use it as-is
            chunks.append(para)
        else:
            # Paragraph too long, split by sentences
            sentences = sent_tokenize(para)
            current_chunk = []
            current_tokens = 0
            
            for sent in sentences:
                sent_tokens = len(tokenizer.encode(sent, add_special_tokens=False))
                
                if current_tokens + sent_tokens > max_tokens and current_chunk:
                    chunks.append(' '.join(current_chunk))
                    current_chunk = [sent]
                    current_tokens = sent_tokens
                else:
                    current_chunk.append(sent)
                    current_tokens += sent_tokens
            
            if current_chunk:
                chunks.append(' '.join(current_chunk))
    
    return chunks

# Test semantic chunking
semantic_chunks = semantic_chunking(rag_document, max_tokens=150)

print(f"Semantic Chunking Results:\n")
print(f"Created {len(semantic_chunks)} semantic chunks:\n")

for i, chunk in enumerate(semantic_chunks, 1):
    print(f"Chunk {i} ({len(chunk)} chars):")
    print(chunk[:120] + "..." if len(chunk) > 120 else chunk)
    print()

print("💡 Semantic chunking preserves paragraph structure!")

Semantic Chunking Results:

Created 8 semantic chunks:

Chunk 1 (51 chars):
Title: Understanding Retrieval-Augmented Generation

Chunk 2 (166 chars):
Retrieval-Augmented Generation (RAG) is a powerful technique in modern NLP. It was 
introduced by Facebook AI Research i...

Chunk 3 (209 chars):
First, the system retrieves relevant documents from a knowledge base. This is typically 
done using dense embeddings and...

Chunk 4 (199 chars):
Second, the retrieved documents serve as context for a language model. Models like GPT-3.5, 
GPT-4, or T5 then generate ...

Chunk 5 (179 chars):
The benefits are clear: reduced hallucination, better factual accuracy, and the ability to 
update knowledge without ret...

Chunk 6 (208 chars):
For implementation, start with document chunking. Use sentence-based chunking with overlap. 
Embed each chunk using a su...

Chunk 7 (164 chars):
At query time, embed the user's question. Retrieve top-k similar chunks. Pass them to the 
LLM as context. The LLM gener

## 11. Chunking Strategies Comparison

In [13]:
# Compare different chunking strategies
test_doc = rag_document[:500]  # Use portion for clarity

strategies = {
    "Fixed sentences (3)": lambda t: [' '.join(sent_tokenize(t)[i:i+3]) 
                                      for i in range(0, len(sent_tokenize(t)), 3)],
    "Token-based (100)": lambda t: chunk_by_sentences(t, max_tokens=100),
    "With overlap (100, 1)": lambda t: chunk_with_overlap(t, max_tokens=100, overlap_sentences=1),
    "Semantic": lambda t: semantic_chunking(t, max_tokens=100),
}

print("Chunking Strategy Comparison:\n")
print(f"Document: {test_doc[:100]}...\n")
print(f"{'Strategy':<25} {'Chunks':<10} {'Avg Length'}")
print("="*60)

for name, strategy in strategies.items():
    chunks = strategy(test_doc)
    avg_len = sum(len(c) for c in chunks) / len(chunks) if chunks else 0
    print(f"{name:<25} {len(chunks):<10} {avg_len:.1f}")

print("\n💡 Choose strategy based on your RAG use case!")

Chunking Strategy Comparison:

Document: Title: Understanding Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) is a power...

Strategy                  Chunks     Avg Length
Fixed sentences (3)       3          165.3
Token-based (100)         1          498.0
With overlap (100, 1)     2          283.0
Semantic                  4          123.5

💡 Choose strategy based on your RAG use case!


## Key Takeaways 🎯

### ✅ Why Sentence Tokenization Matters for RAG:

1. **Semantic Units**: Sentences preserve meaning better than random chunks
2. **Token Limits**: Must fit within model context windows (512-4096 tokens)
3. **Retrieval Quality**: Better granularity = more precise context
4. **Answer Accuracy**: Relevant sentences = better LLM responses

### 🔧 Best Tools:

```python
# For English:
from nltk.tokenize import sent_tokenize  # ✅ Best

# For multiple languages:
import spacy  # ✅ Also great

# Avoid:
text.split('.')  # ❌ Too naive
```

### 📊 Chunking Strategies for RAG:

| Strategy | Best For | Pros | Cons |
|----------|----------|------|------|
| **Sentence-based** | General RAG | Preserves meaning | May exceed token limit |
| **Token-limited** | Fixed context | Respects limits | May cut sentences |
| **With overlap** | Context preservation | Better retrieval | More chunks |
| **Semantic** | Structured docs | Natural boundaries | Complex logic |

### 🎯 Recommended Approach for RAG:

```python
# BEST PRACTICE:
def rag_chunking(document, max_tokens=512, overlap=2):
    """
    1. Split by sentences (NLTK)
    2. Group into token-limited chunks
    3. Add overlap for context
    4. Embed each chunk
    5. Store in vector DB
    """
    sentences = sent_tokenize(document)
    chunks = chunk_with_overlap(
        document, 
        max_tokens=max_tokens,
        overlap_sentences=overlap
    )
    return chunks
```

### 💡 Pro Tips:

1. **Chunk Size**:
   - Too small: Lacks context
   - Too large: Exceeds limits, noisy retrieval
   - Sweet spot: 256-512 tokens

2. **Overlap**:
   - Use 1-2 sentence overlap
   - Helps with boundary cases
   - Improves retrieval recall

3. **Preprocessing**:
   - Clean text first (remove extra spaces)
   - Handle special characters
   - Preserve structure (headers, lists)

4. **Testing**:
   - Test on your domain's documents
   - Measure retrieval quality
   - Adjust chunk size as needed

### 🚀 Production RAG Pipeline:

```python
# Complete pipeline
from nltk.tokenize import sent_tokenize
from transformers import AutoTokenizer, AutoModel

# 1. Chunk document
chunks = chunk_with_overlap(document, max_tokens=512)

# 2. Embed chunks
embeddings = model.encode(chunks)

# 3. Store in vector DB
vector_db.add(chunks, embeddings)

# 4. At query time:
query_embedding = model.encode(query)
relevant_chunks = vector_db.search(query_embedding, top_k=5)

# 5. Generate with LLM
context = '\n'.join(relevant_chunks)
response = llm.generate(context, query)
```

### ⚠️ Common Pitfalls:

1. **Ignoring abbreviations**: Use proper tokenizer (NLTK/spaCy)
2. **No overlap**: Chunks miss boundary context
3. **Wrong chunk size**: Test and optimize for your use case
4. **Losing structure**: Preserve headers, paragraphs when possible

## Next Steps 🔜

You've mastered tokenization! Now you understand:
- ✅ Word, character, subword tokenization
- ✅ BPE, WordPiece, SentencePiece
- ✅ Sentence tokenization for RAG chunking

**Ready to move to the next folder: `2_retrieval/`!** 🚀

That's where we'll build the retrieval system using these tokenization techniques!